Instalación de paquetes e importación de librerías

In [1]:
!pip install graphviz 
!pip install semopy

import numpy as np
import pandas as pd
import semopy
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats



Preparación de datos y creación de dataframe

In [2]:
df_raw = pd.read_csv("C:/Users/Usuario/Downloads/Datos brutos ESS.csv",
                    sep=";", decimal=",", low_memory=False)

cols = ["happy", "stflife", "wrhpp", "enjlf", "fltsd", "fltdpr", "fltlnl", "cldgng",
        "mnactic", "hincfel", "wkhtot", "wrkctra", "wkdcorga", "jbspv",
        "nobingnd", "agegroup"]
base = df_raw[cols].apply(pd.to_numeric, errors="coerce")

Realización de Análisis Factorial Confirmatorio

In [3]:
### Preparación de datos AFC ###

base[["happy", "stflife"]] = base[["happy", "stflife"]].replace([77, 88, 99], np.nan)

afecto = ["wrhpp", "enjlf", "fltsd", "fltdpr", "fltlnl", "cldgng"]
base[afecto] = base[afecto].replace([7, 8, 9], np.nan)

variables_invertir = ["fltsd", "fltdpr", "fltlnl", "cldgng"]
for variable in variables_invertir: 
    base[f"{variable}_inv"] = 5 - base [variable]
base = base.drop(columns=variables_invertir)
afecto_cfa = ["wrhpp", "enjlf", "fltsd_inv", "fltdpr_inv", "fltlnl_inv", "cldgng_inv"]

vars_cfa = ["happy", "stflife"] + afecto_cfa
df_cfa = base[vars_cfa].dropna().reset_index(drop=True)
print(f"CFA | filas iniciales: {len(base)} | eliminadas: {len(base) - len(df_cfa)} | N final: {len(df_cfa)}")


### Planteamiento de los modelos ###

modelo_A = """
AP =~ wrhpp + enjlf
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ stflife + cldgng_inv
SWB =~ AP + AN + SV
happy ~ SWB
"""
modelo_B = """
AP =~ wrhpp + enjlf + happy
AN =~ fltsd_inv + fltdpr_inv + fltlnl_inv
SV =~ cldgng_inv
SWB =~ AP + AN + SV
stflife ~ SWB
"""


### Análisis Factorial Confirmatorio ###

mA = semopy.Model(modelo_A); mA.fit(df_cfa)
mB = semopy.Model(modelo_B); mB.fit(df_cfa)

def tabla_estimaciones(modelo, titulo):
    res = modelo.inspect(std_est=True).copy()
    res["p-value"] = pd.to_numeric(res["p-value"], errors="coerce")
    res["p-value"] = res["p-value"].apply(
        lambda p: "-" 
            if pd.isna(p) 
            else ("<.001" if p < 0.001 else f"{p:.3f}")
        )
    print(f"\n{titulo}")
    print(res.round(4).to_string())

tabla_estimaciones(mA, "MODELO A: estimaciones")
tabla_estimaciones(mB, "MODELO B: estimaciones")

sA = semopy.calc_stats(mA).iloc[0]
sB = semopy.calc_stats(mB).iloc[0]
tabla = pd.DataFrame({
    "A (happy como SWB)": [sA["chi2"], sA["DoF"], sA["CFI"], sA["TLI"], sA["RMSEA"]],
    "B (stflife como SWB)": [sB["chi2"], sB["DoF"], sB["CFI"], sB["TLI"], sB["RMSEA"]],
}, index=["chi2", "gl", "CFI", "TLI", "RMSEA"]).astype(float)
print("\nÍNDICES DE AJUSTE: COMPARACIÓN DE MODELOS")
print(tabla.round(3))

d_chi = sB["chi2"] - sA["chi2"]
d_gl = sB["DoF"] - sA["DoF"]
print(f"\nB vs A: Δχ² = {d_chi:.2f}, Δgl = {d_gl:.0f}, p = {stats.chi2.sf(d_chi, d_gl):.4g}")


CFA | filas iniciales: 50116 | eliminadas: 1617 | N final: 48499

MODELO A: estimaciones
          lval  op        rval  Estimate  Est. Std  Std. Err     z-value p-value
0           AP   ~         SWB    1.0000    0.7290         -           -       -
1           AN   ~         SWB    0.7815    0.7119  0.007794  100.269228   <.001
2           SV   ~         SWB    3.2917    1.0000  0.027252  120.787622   <.001
3        wrhpp   ~          AP    1.0000    0.8268         -           -       -
4        enjlf   ~          AP    0.9975    0.7846  0.007185  138.832428   <.001
5    fltsd_inv   ~          AN    1.0000    0.7730         -           -       -
6   fltdpr_inv   ~          AN    0.9120    0.7164  0.006807  133.984325   <.001
7   fltlnl_inv   ~          AN    0.8988    0.6628  0.007095  126.676012   <.001
8      stflife   ~          SV    1.0000    0.7552         -           -       -
9   cldgng_inv   ~          SV    0.2022    0.4510  0.002251   89.860683   <.001
10       happy   ~  

Regresiones lineales

In [4]:
n_inicial = len(base)


### Preparación y limpieza de las variables para el dataframe ###

df = base[base["mnactic"] == 1]
n_trabajadores = len(df)

# Acotar para coger los valores de interés
df = df[df["happy"] <= 10]
df = df[df["hincfel"].isin([1, 2, 3, 4])]
df = df[df["wkhtot"].between(1, 80)]
df = df[df["wrkctra"].isin([1, 2, 3])]
df = df[df["wkdcorga"] <= 10]
df = df[df["jbspv"].isin([1, 2])]
df = df[df["nobingnd"].isin([1, 2, 3])]
df = df[df["agegroup"].between(1, 7)]

df["hincfel_inv"] = 5 - df["hincfel"]

vars_reg = ["happy", "hincfel_inv", "wkhtot", "wrkctra", "wkdcorga", "jbspv", "nobingnd", "agegroup"]
df = df.dropna(subset=vars_reg).reset_index(drop=True)

# Cálculo de instancias con las que trabajar
print(f"\nRegresión | filas iniciales: {n_inicial}")
print(f"Trabajadores remunerados: {n_trabajadores}")
print(f"Eliminados por missing/no válidos: {n_trabajadores - len(df)}")
print(f"Muestra final: {len(df)}")

# Recodificación de nombres simples
df["genero"] = df["nobingnd"]
df["edad"] = df["agegroup"]
df["jbspv_r"] = (df["jbspv"] == 1).astype(int)
df["wrkctra_limited"] = (df["wrkctra"] == 2).astype(int)
df["wrkctra_nocontract"] = (df["wrkctra"] == 3).astype(int)
for v in ["hincfel_inv", "wkhtot", "wkdcorga"]:
    df[v + "_c"] = df[v] - df[v].mean()

VIS = "hincfel_inv_c + wkhtot_c + wrkctra_limited + wrkctra_nocontract + wkdcorga_c + jbspv_r"


### Modelo 2 Regresión sin ponderación ###

m_nomod = smf.ols(f"happy ~ {VIS}", data=df).fit()
print("\n\n::::::::::::::::::::::::::::::::\nRegresión sin moderación\n::::::::::::::::::::::::::::::::\n")
print(m_nomod.summary())


### Modelo 3 regresión con ponderaciones

# Regresión con ponderación de género
m_modgen = smf.ols(f"happy ~ {VIS} + C(genero)", data=df).fit()
m_modgen_interacc = smf.ols(f"happy ~ ({VIS}) * C(genero)", data=df).fit()
print("\n\n:::::::::::::::::::::::::::::::::::::::::\nRegresión con moderación por género (3A)\n:::::::::::::::::::::::::::::::::::::::::\n")
print(m_modgen_interacc.summary())
print("\nTest de interacciones 3A (género):\n-----------------------------------")
print(anova_lm(m_modgen, m_modgen_interacc))

# Regresión con ponderación de edad
m_moded = smf.ols(f"happy ~ {VIS} + C(edad)", data=df).fit()
m_moded_interacc = smf.ols(f"happy ~ ({VIS}) * C(edad)", data=df).fit()
print("\n\n:::::::::::::::::::::::::::::::::::::::::\nRegresión con moderación por edad (3B)\n:::::::::::::::::::::::::::::::::::::::::\n")
print(m_moded_interacc.summary())
print("\nTest de interacciones 3B (edad):\n-----------------------------------")
print(anova_lm(m_moded, m_moded_interacc))

# Regresión con ponderación de género y edad
m_modgened = smf.ols(f"happy ~ {VIS} + C(genero) + C(edad)", data=df).fit()
m_modgened_interacc = smf.ols(
    f"happy ~ {VIS} + C(genero) + C(edad) + ({VIS}):C(genero) + ({VIS}):C(edad)",
    data=df).fit()
print("\n\n:::::::::::::::::::::::::::::::::::::::::::::::::\nRegresión con moderación por género y edad (3C)\n:::::::::::::::::::::::::::::::::::::::::::::::::\n")
print(m_modgened_interacc.summary())
print("\nTest de interacciones 3C (género y edad):\n-------------------------------------------")
print(anova_lm(m_modgened, m_modgened_interacc))   # orden corregido: sin interacciones primero

# Comparación de regresiones ponderadas
print("\n\n:::::::::::::::::::::::::::::::::::::::::::::::::\nComparación de 3 modelos\n:::::::::::::::::::::::::::::::::::::::::::::::::\n")
print(pd.DataFrame({
    "Modelo": ["3A género", "3B edad", "3C conjunto"],
    "R2": [m_modgen_interacc.rsquared, m_moded_interacc.rsquared, m_modgened_interacc.rsquared],
    "R2 ajustado": [m_modgen_interacc.rsquared_adj, m_moded_interacc.rsquared_adj,
                    m_modgened_interacc.rsquared_adj],
}).round(3))



Regresión | filas iniciales: 50116
Trabajadores remunerados: 25677
Eliminados por missing/no válidos: 5570
Muestra final: 20107


::::::::::::::::::::::::::::::::
Regresión sin moderación
::::::::::::::::::::::::::::::::

                            OLS Regression Results                            
Dep. Variable:                  happy   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     384.6
Date:                Sat, 07 Nov 2026   Prob (F-statistic):               0.00
Time:                        16:57:56   Log-Likelihood:                -38044.
No. Observations:               20107   AIC:                         7.610e+04
Df Residuals:                   20100   BIC:                         7.616e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                             